# 5.2 pandas → PySpark Translation

Side-by-side reference for translating common pandas idioms to PySpark DataFrame API. Each pattern is shown as pandas (runs locally) with the PySpark equivalent annotated.

Patterns:
1. Read CSV
2. Select columns
3. Filter rows
4. Add a derived column
5. GroupBy + aggregate
6. Sort
7. Join / merge
8. Drop duplicates
9. Handle nulls
10. String operations
11. Date arithmetic
12. Window functions
13. Pivot / reshape

Plus three SAP-specific examples at the end.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

csv = StringIO('''MATNR,WERKS,LBKUM,VPRSV,STPRS,BUDAT
MAT-00001,1000,120,S,15.50,2025-09-01
MAT-00001,2000,80,S,15.50,2025-09-15
MAT-00002,1000,40,V,220.00,2025-09-10
MAT-00003,2000,0,S,5.25,2025-08-20
MAT-00004,1000,200,V,3.10,2025-09-22
''')
df = pd.read_csv(csv, parse_dates=['BUDAT'])
df

## 1. Read CSV

In [ ]:
# pandas
# df = pd.read_csv('inventory.csv', parse_dates=['BUDAT'])
#
# PySpark
# df = (spark.read.option('header', True).option('inferSchema', True)
#       .csv('/Volumes/raw/sap/inventory.csv'))

## 2. Select columns

In [ ]:
df[['MATNR', 'WERKS', 'LBKUM']].head()
# PySpark: df.select('MATNR', 'WERKS', 'LBKUM')

## 3. Filter rows

In [ ]:
df[df['LBKUM'] > 0]
# PySpark: df.filter(F.col('LBKUM') > 0)

## 4. Add a derived column

In [ ]:
df = df.assign(stock_value=df['LBKUM'] * df['STPRS'])
df.head()
# PySpark: df.withColumn('stock_value', F.col('LBKUM') * F.col('STPRS'))

## 5. GroupBy + aggregate

In [ ]:
df.groupby('WERKS').agg(total_qty=('LBKUM', 'sum'), n_mat=('MATNR', 'nunique'))
# PySpark:
# df.groupBy('WERKS').agg(
#     F.sum('LBKUM').alias('total_qty'),
#     F.countDistinct('MATNR').alias('n_mat'))

## 6. Sort

In [ ]:
df.sort_values(['WERKS', 'LBKUM'], ascending=[True, False]).head()
# PySpark: df.orderBy('WERKS', F.col('LBKUM').desc())

## 7. Join / merge

In [ ]:
plants = pd.DataFrame({'WERKS': ['1000', '2000'], 'plant_name': ['Hamburg', 'Munich']})
df.merge(plants, on='WERKS', how='left').head()
# PySpark: df.join(plants, on='WERKS', how='left')

## 8. Drop duplicates

In [ ]:
df.drop_duplicates(subset=['MATNR'])
# PySpark: df.dropDuplicates(['MATNR'])

## 9. Handle nulls

In [ ]:
df.fillna({'LBKUM': 0}).dropna(subset=['MATNR'])
# PySpark: df.fillna({'LBKUM': 0}).dropna(subset=['MATNR'])

## 10. String operations

In [ ]:
df['MATNR'].str.startswith('MAT-0000').head()
# PySpark: df.withColumn('flag', F.col('MATNR').startswith('MAT-0000'))

## 11. Date arithmetic

In [ ]:
df['days_since'] = (pd.Timestamp('2025-10-01') - df['BUDAT']).dt.days
df[['MATNR', 'BUDAT', 'days_since']].head()
# PySpark: df.withColumn('days_since', F.datediff(F.lit('2025-10-01'), F.col('BUDAT')))

## 12. Window functions

In [ ]:
df['rank_in_plant'] = df.groupby('WERKS')['LBKUM'].rank(ascending=False, method='dense')
df[['MATNR', 'WERKS', 'LBKUM', 'rank_in_plant']]
# PySpark:
# from pyspark.sql.window import Window
# w = Window.partitionBy('WERKS').orderBy(F.col('LBKUM').desc())
# df.withColumn('rank_in_plant', F.dense_rank().over(w))

## 13. Pivot / reshape

In [ ]:
df.pivot_table(index='MATNR', columns='WERKS', values='LBKUM', aggfunc='sum', fill_value=0)
# PySpark: df.groupBy('MATNR').pivot('WERKS').sum('LBKUM')

---
## SAP-specific examples

### A. Inventory aging buckets (MMBE-style)

In [ ]:
today = pd.Timestamp('2025-10-01')
df['age_days'] = (today - df['BUDAT']).dt.days
df['age_bucket'] = pd.cut(df['age_days'], [-1, 30, 60, 90, 9999],
                          labels=['0-30', '31-60', '61-90', '90+'])
df[['MATNR', 'age_days', 'age_bucket']]
# PySpark equivalent uses F.when(...).otherwise(...) chain on F.datediff.

### B. Cost variance vs plan (KSB1-style)

In [ ]:
actuals = pd.DataFrame({'KOSTL': ['CC-1001', 'CC-1002'], 'actual': [12000, 8500]})
plan = pd.DataFrame({'KOSTL': ['CC-1001', 'CC-1002'], 'plan': [10000, 9000]})
var = actuals.merge(plan, on='KOSTL').assign(variance=lambda x: x['actual'] - x['plan'])
var
# PySpark: actuals.join(plan, 'KOSTL').withColumn('variance', F.col('actual') - F.col('plan'))

### C. BW-style fact rollup (revenue × margin)

In [ ]:
fact = pd.DataFrame({
    'CALMONTH': ['2025-08', '2025-08', '2025-09'],
    'KUNNR': ['CUST-0001', 'CUST-0002', 'CUST-0001'],
    'revenue': [10000, 5000, 12000],
    'cogs':    [6500,  3200,  7800]
})
fact.assign(margin=lambda x: x['revenue'] - x['cogs']) \
    .groupby('CALMONTH').agg(rev=('revenue', 'sum'), mgn=('margin', 'sum'))
# PySpark equivalent: withColumn('margin', ...).groupBy('CALMONTH').agg(F.sum(...), F.sum(...))